# 🎨 TEST - Animagine XL 3.1 (5 personajes de prueba)
Ejecuta este notebook en Google Colab con GPU T4 para ver cómo quedarán los avatares en estilo anime antes de lanzar el mega script.

In [ ]:
# Celda 1: Instalación
!pip install -q diffusers transformers accelerate torch torchvision

In [ ]:
# Celda 2: Subir el JSON directamente (sin Drive)
from google.colab import files
print('Selecciona tu archivo characters_to_generate.json')
uploaded = files.upload()
print('✅ Archivo subido!')

In [ ]:
# Celda 3: Cargar modelo y generar 5 personajes de prueba
import json
import torch
import os
from diffusers import StableDiffusionXLPipeline
from IPython.display import display, Image as IPImage
import zipfile

# Leer JSON y tomar 5 personajes de ejemplo (distintos libros)
with open('characters_to_generate.json', 'r', encoding='utf-8') as f:
    characters = json.load(f)

print(f'Total de personajes en el catálogo: {len(characters)}')

# Tomar 5 de distintas partes del JSON para mejor muestra
step = len(characters) // 5
samples = [characters[i * step] for i in range(5)]

print(f'\nPersonajes seleccionados para la prueba:')
for i, c in enumerate(samples):
    print(f'  {i+1}. {c["name"]} ({c["book"]})')

# Cargar el motor anime
MODEL_ID = 'cagliostrolab/animagine-xl-3.1'
print(f'\n🚀 Cargando modelo {MODEL_ID} (~7GB, puede tardar 3-4 min)...')

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

print('\n🎨 Generando imágenes de prueba...')
os.makedirs('test_output', exist_ok=True)

quality_tags = 'masterpiece, best quality, very aesthetic, absurdres, highres, intricate details'
negative_prompt = 'lowres, (bad), text, error, fewer, extra, missing, worst quality, jpeg artifacts, low quality, watermark, unfinished, displeasing, oldest, early, chromatic aberration, signature, extra digits, artistic error, username, scan, [abstract]'

for i, char in enumerate(samples):
    print(f'  Generando ({i+1}/5): {char["name"]} de "{char["book"]}"...')
    
    prompt = char.get('prompt', 'portrait of a character')
    enhanced_prompt = f'{prompt}, {quality_tags}'
    
    image = pipe(
        prompt=enhanced_prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=28,
        guidance_scale=7.0,
        width=832,
        height=1216
    ).images[0]
    
    filename = f'test_output/{i+1}_{char["name"][:30].replace(" ", "_")}.png'
    image.save(filename)
    print(f'    ✅ Guardada: {filename}')
    display(IPImage(filename, width=300))

# Crear ZIP de prueba para descargar
with zipfile.ZipFile('test_anime_preview.zip', 'w') as zipf:
    for f in os.listdir('test_output'):
        zipf.write(os.path.join('test_output', f), f)

print('\n📦 ZIP de prueba creado: test_anime_preview.zip')
print('🎉 ¡Prueba completada! Descarga el ZIP abajo para revisar las 5 imágenes.')

In [ ]:
# Celda 4: Descargar el ZIP de prueba
from google.colab import files
files.download('test_anime_preview.zip')
print('✅ Descarga iniciada!')